<a href="https://colab.research.google.com/github/va4756/bigdata_raejung/blob/main/LLM_product_chap2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2-6 RNN과 LSTM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install gensim
!python -m spacy download en_core_web_lg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 39.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
import nltk
import spacy

In [ ]:
# nltk.download('stopwords')
# tokenizer = nltk.tokenize.RegexpTokenizer(r"\w+'?\w+|\w+'")
# tokenizer.tokenize("I'm studying NLP and it's interesting.")
# stop_words = nltk.corpus.stopwords.words("english")
# print(stop_words[:20])
# print(len(stop_words))

In [ ]:
try:
    tokenizer = nltk.tokenize.RegexpTokenizer(r"\w+'?\w+|\w+'")
    tokenizer.tokenize("I'm studying NLP and it's interesting.")
    stop_words = nltk.corpus.stopwords.words("english")
    nlp = spacy.load("en_core_web_lg", disable=["parser", "ner"]) # disable=["parser", "tagger", "ner"]
except Exception:
    nltk.download("stopwords")
    nltk.download("punkt")
    spacy.cli.download("en_core_web_lg")
    tokenizer = nltk.tokenize.RegexpTokenizer(r"\w+'?\w+|\w+'")
    tokenizer.tokenize("I'm studying NLP and it's interesting.")
    stop_words = nltk.corpus.stopwords.words("english")
    nlp = spacy.load("en_core_web_lg", disable=["parser", "ner"])  # disable=["parser", "tagger", "ner"]
    print(stop_words[:20])
    print(len(stop_words))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']
198


In [ ]:
# Create our corpus for training and perform some classic NLP preprocessing
dataset = pd.read_csv("/content/twitter.csv")

In [ ]:
text_data = list(
    map(lambda x: tokenizer.tokenize(x.lower()), dataset['text'])
)
text_data = [
    [token.lemma_ for word in text for token in nlp(word)] for text in text_data
]
label_data = list(map(lambda x: x, dataset["feeling"]))
assert len(text_data) == len(label_data), f"{len(text_data)} does not equal {len(label_data)}"

In [ ]:
EMBEDDING_DIM = 100
model = Word2Vec(
    text_data, vector_size=EMBEDDING_DIM, window=5, min_count=1, workers=4
)
word_vectors = model.wv
print(f"Vocabulary Length: {len(model.wv)}")
del model

Vocabulary Length: 626


In [ ]:
padding_value = len(word_vectors.index_to_key)
# Embeddings are needed to give semantic value to the inputs of an LSTM
embedding_weights = torch.Tensor(word_vectors.vectors)

In [ ]:
# RNN model

class RNN(nn.Module):

    def __init__(self,
                 input_dim,
                 embedding_dim,
                 hidden_dim,
                 output_dim,
                 n_layers,
                 bidirectional,
                 dropout,
                 embedding_weights):

        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights
        )

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout
        )

        self.fc = nn.Linear(
            hidden_dim * 2 if bidirectional else hidden_dim,
            output_dim
        )

        # 추가 필요
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, text_lengths):

        embedded = self.embedding(x)

        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded,
            text_lengths.cpu(),
            enforce_sorted=False
        )

        packed_output, hidden = self.rnn(packed_embedded)

        output, output_lengths = nn.utils.rnn.pad_packed_sequence(
            packed_output
        )

        if self.rnn.bidirectional:
            hidden = self.dropout(
                torch.cat((hidden[-2], hidden[-1]), dim=1)
            )
        else:
            hidden = self.dropout(hidden[-1])

        return self.fc(hidden)

# class RNN(nn.Module):
#     def __init__(
#             self,
#             input_dim,
#             embedding_dim,
#             hidden_dim,
#             output_dim,
#             embedding_weights
#     ):
#         super().__init__()
#         self.embedding = nn.Embedding.from_pretrained(embedding_weights)
#         self.rnn = nn.RNN(embedding_dim, hidden_dim)
#         self.fc = nn.Linear(hidden_dim, output_dim)

#     def forward(self, x, text_lengths):

#         embedded = self.embedding(x)

#         packed_embedded = nn.utils.rnn.pack_padded_sequence(
#             embedded,
#             text_lengths.cpu(),
#             enforce_sorted=False
#         )

#         packed_output, hidden = self.rnn(packed_embedded)

#         output, output_lengths = nn.utils.rnn.pad_packed_sequence(
#             packed_output
#         )

#         hidden = self.dropout(hidden[-1])

#         return self.fc(hidden)

    # def forward(self, x, text_lengths):
    #     embedded = self.embedding(x)
    #     packed_embedded = nn.utils.rnn.pad_packed_sequence(embedded, text_lengths)
    #     packed_output, hidden = self.rnn(packed_embedded)
    #     output, output_lengths = nn.utils.rnn.pad_packed_sequence(packed_output)

    #     return self.fc(hidden.squeeze(0))

In [ ]:
INPUT_DIM = padding_value
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1

In [ ]:
rnn_model = RNN(
    INPUT_DIM,
    EMBEDDING_DIM,
    HIDDEN_DIM,
    OUTPUT_DIM,
    N_LAYERS,
    BIDIRECTIONAL,
    DROPOUT,
    embedding_weights
)

In [ ]:
rnn_optimizer = optim.SGD(rnn_model.parameters(), lr=1e-3)
rnn_criterion = nn.BCEWithLogitsLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# LSTM model
class LSTM(nn.Module):
    def __init__(self,
                 input_dim,
                 embedding_dim,
                 hidden_dim,
                 output_dim,
                 n_layers,
                 bidirectional,
                 dropout,
                 embedding_weights):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_weights)
        self.rnn = nn.LSTM(embedding_dim,
                           hidden_dim,
                           num_layers=n_layers,
                           bidirectional=bidirectional,
                           dropout=dropout)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, text_lengths):

        embedded = self.embedding(x)

        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded,
            text_lengths.cpu(),
            enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.rnn(packed_embedded)

        output, output_lengths = nn.utils.rnn.pad_packed_sequence(
            packed_output
        )

        hidden = self.dropout(
            torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        )

        return self.fc(hidden)

    # def forward(self, x, text_lengths):
    #     embedded = self.embedding(x)
    #     packed_embedded = nn.utils.rnn.pack_padded_sequence(
    #         embedded, text_lengths
    #     )
    #     packed_output, (hidden, cell) = self.rnn(packed_embedded)
    #     hidden = self.dropout(
    #         torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
    #     )
    #     return self.fc(hidden.squeeze(0))

In [ ]:
INPUT_DIM = padding_value
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5

In [ ]:
lstm_model = LSTM(INPUT_DIM,
                  EMBEDDING_DIM,
                  HIDDEN_DIM,
                  OUTPUT_DIM,
                  N_LAYERS,
                  BIDIRECTIONAL,
                  DROPOUT,
                  embedding_weights)

In [ ]:
lstm_optimizer = optim.Adam(lstm_model.parameters())
lstm_criterion = nn.BCEWithLogitsLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def binary_accuracy(preds, y):
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    acc = correct.sum() / len(correct)
    return acc

In [ ]:
def train(model, iterator, optimizer, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.train()
    for batch in iterator:
        optimizer.zero_grad()
        predictions = model(batch["text"], batch["length"]).squeeze(1)
        loss = criterion(predictions, batch["label"])
        acc = binary_accuracy(predictions, batch["label"])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
def evaluate(model, iterator, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            predictions = model(batch["text"], batch["length"]).squeeze(1)
            loss = criterion(predictions, batch["label"])
            acc = binary_accuracy(predictions, batch["label"])

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
# Usually should be a power of 2 because it's the easiest for computer memory.
batch_size = 2

In [ ]:
def iterator(X, y):
    size = len(X)
    permutation = np.random.permutation(size)
    iterate = []
    for i in range(0, size, batch_size):
        indices = permutation[i : i + batch_size]
        batch = {}
        batch["text"] = [X[i] for i in indices]
        batch["label"] = [y[i] for i in indices]

        batch["text"], batch["label"] = zip(
            *sorted(
                zip(batch["text"], batch["label"]),
                key=lambda x: len(x[0]),
                reverse=True,
            )
        )
        batch["length"] = [len(utt) for utt in batch["text"]]
        batch["length"] = torch.IntTensor(batch["length"])
        batch["text"] = torch.nn.utils.rnn.pad_sequence(
            batch["text"], batch_first=True
        ).t()
        batch["label"] = torch.Tensor(batch["label"])

        batch["label"] = batch["label"].to(device)
        batch["length"] = batch["length"].to(device)
        batch["text"] = batch["text"].to(device)

        iterate.append(batch)

    return iterate

In [ ]:
index_utt = [
    torch.tensor([word_vectors.key_to_index.get(word, 0) for word in text])
    for text in text_data
]

In [ ]:
# You've got to determine some labels for whatever you're training on.
X_train, X_test, y_train, y_test = train_test_split(
    index_utt, label_data, test_size=0.2
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2
)

In [ ]:
train_iterator = iterator(X_train, y_train)
validate_iterator = iterator(X_val, y_val)
test_iterator = iterator(X_test, y_test)

print(len(train_iterator), len(validate_iterator), len(test_iterator))

32 8 10


In [ ]:
N_EPOCHS = 25

In [ ]:
for model in [rnn_model, lstm_model]:
    print(
        "|-----------------------------------------------------------------------------------------|"
    )
    print(f"Training with {model.__class__.__name__}")
    if "RNN" in model.__class__.__name__:
        for epoch in range(N_EPOCHS):
            train_loss, train_acc = train(
                rnn_model, train_iterator, rnn_optimizer, rnn_criterion
            )
            valid_loss, valid_acc = evaluate(
                rnn_model, validate_iterator, rnn_criterion
            )

            print(
                f"| Epoch: {epoch+1:02} | Train Loss: {train_loss: .3f} | Train Acc: {train_acc*100: .2f}% | Validation Loss: {valid_loss: .3f} | Validation Acc: {valid_acc*100: .2f}% |"
            )
    else:
        for epoch in range(N_EPOCHS):
            train_loss, train_acc = train(
                lstm_model, train_iterator, lstm_optimizer, lstm_criterion
            )
            valid_loss, valid_acc = evaluate(
                lstm_model, validate_iterator, lstm_criterion
            )

            print(
                f"| Epoch: {epoch+1:02} | Train Loss: {train_loss: .3f} | Train Acc: {train_acc*100: .2f}% | Validation Loss: {valid_loss: .3f} | Validation Acc: {valid_acc*100: .2f}% |"
            )

|-----------------------------------------------------------------------------------------|
Training with RNN
| Epoch: 01 | Train Loss:  0.695 | Train Acc:  48.44% | Validation Loss:  0.693 | Validation Acc:  50.00% |
| Epoch: 02 | Train Loss:  0.697 | Train Acc:  45.31% | Validation Loss:  0.693 | Validation Acc:  56.25% |
| Epoch: 03 | Train Loss:  0.696 | Train Acc:  46.88% | Validation Loss:  0.692 | Validation Acc:  56.25% |
| Epoch: 04 | Train Loss:  0.703 | Train Acc:  42.19% | Validation Loss:  0.692 | Validation Acc:  56.25% |
| Epoch: 05 | Train Loss:  0.696 | Train Acc:  51.56% | Validation Loss:  0.691 | Validation Acc:  56.25% |
| Epoch: 06 | Train Loss:  0.702 | Train Acc:  40.62% | Validation Loss:  0.691 | Validation Acc:  56.25% |
| Epoch: 07 | Train Loss:  0.688 | Train Acc:  54.69% | Validation Loss:  0.690 | Validation Acc:  56.25% |
| Epoch: 08 | Train Loss:  0.692 | Train Acc:  59.38% | Validation Loss:  0.690 | Validation Acc:  56.25% |
| Epoch: 09 | Train Loss: 

## 2-7 Attention

In [95]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [96]:
import numpy as np
from scipy.special import softmax, logsumexp

In [97]:
print("Step 1: Input : 3 inputs, d_model=4")
x = np.array(              # 4x3 matrix
    [[1.0, 0.0, 1.0, 0.0],
     [0.0, 2.0, 0.0, 2.0],
     [1.0, 1.0, 1.0, 1.0]]
)
print("x:", x)

Step 1: Input : 3 inputs, d_model=4
x: [[1. 0. 1. 0.]
 [0. 2. 0. 2.]
 [1. 1. 1. 1.]]


In [98]:
print("Step 2: weights 3 dimensions x d_model=4")
w_query = np.array([[1, 0, 1],    # 3x4 matrix
                    [1, 0, 0],
                    [0, 0, 1],
                    [0, 1, 1]])
print("w_query:", w_query)

Step 2: weights 3 dimensions x d_model=4
w_query: [[1 0 1]
 [1 0 0]
 [0 0 1]
 [0 1 1]]


In [99]:
w_key = np.array([[0, 0, 1],
                 [1, 1, 0],
                 [0, 1, 0],
                 [1, 1, 0]])
print("w_key:", w_key)

w_key: [[0 0 1]
 [1 1 0]
 [0 1 0]
 [1 1 0]]


In [100]:
w_value = np.array([[0, 2, 0], [0, 3, 0], [1, 0, 3], [1, 1, 0]])
print("w_value:", w_value)

w_value: [[0 2 0]
 [0 3 0]
 [1 0 3]
 [1 1 0]]


In [101]:
print("Step 3: Matrix multiplication to obtain Q,K,V")
print("Query: x * w_query")   # (3,4)x(4,3)=(3,3)
Q = np.matmul(x, w_query)
print("Q:", Q)

Step 3: Matrix multiplication to obtain Q,K,V
Query: x * w_query
Q: [[1. 0. 2.]
 [2. 2. 2.]
 [2. 1. 3.]]


In [102]:
print("Key: x * w_key")
K = np.matmul(x, w_key)
print("K:", K)

Key: x * w_key
K: [[0. 1. 1.]
 [4. 4. 0.]
 [2. 3. 1.]]


In [103]:
print("Value: x * w_value")
V = np.matmul(x, w_value)
print("V:", V)

Value: x * w_value
V: [[1. 2. 3.]
 [2. 8. 0.]
 [2. 6. 3.]]


In [104]:
print("Step 4: Scaled Attention Scores")
k_d= 1   # This is normally the square root of the number of dimensions
attention_scores = (Q @ K.transpose()) / k_d
print(attention_scores)

Step 4: Scaled Attention Scores
[[ 2.  4.  4.]
 [ 4. 16. 12.]
 [ 4. 12. 10.]]


In [105]:
print("Step 5: Scaled softmax attention_scores for each vector")
attention_scores[0] = softmax(attention_scores[0])
attention_scores[1] = softmax(attention_scores[1])
attention_scores[2] = softmax(attention_scores[2])
print(attention_scores[0])
print(attention_scores[1])
print(attention_scores[2])

Step 5: Scaled softmax attention_scores for each vector
[0.06337894 0.46831053 0.46831053]
[6.03366485e-06 9.82007865e-01 1.79861014e-02]
[2.95387223e-04 8.80536902e-01 1.19167711e-01]


In [106]:
print("Step 6: attention value obtained by score1/k_d * V")
print(V[0])
print(V[1])
print(V[2])

Step 6: attention value obtained by score1/k_d * V
[1. 2. 3.]
[2. 8. 0.]
[2. 6. 3.]


In [107]:
attention1 = attention_scores[0].reshape(-1, 1)
attention1 = attention_scores[0][0] * V[0]
print("Attention 1:", attention1)

Attention 1: [0.06337894 0.12675788 0.19013681]


In [108]:
attention2 = attention_scores[0][1] * V[1]
print("Attention 2:", attention2)

Attention 2: [0.93662106 3.74648425 0.        ]


In [109]:
attention3 = attention_scores[0][2] * V[2]
print("Attention 3:", attention3)

Attention 3: [0.93662106 2.80986319 1.40493159]


In [110]:
print("Step 7: sum the results to create the first line of the output matrix")
attention_input1 = attention1 + attention2 + attention3
print(attention_input1)

Step 7: sum the results to create the first line of the output matrix
[1.93662106 6.68310531 1.59506841]


In [111]:
print("Step 8: Step 1 to 7 for inputs 1 to 3")
# This assumes that we actually went through the whole process for all 3
# We'll just take a random matrix of the correct dimensions in lieu
attention_head1 = np.random.random((3, 64))
print(attention_head1)

Step 8: Step 1 to 7 for inputs 1 to 3
[[0.48744428 0.46297837 0.03489919 0.26851483 0.15784046 0.24530017
  0.12441016 0.3076796  0.53638692 0.78572521 0.45248431 0.0589916
  0.88838296 0.96636213 0.09866073 0.8110137  0.88333452 0.44010697
  0.39314084 0.63565975 0.59082599 0.83511578 0.37095737 0.92546041
  0.63864808 0.6203146  0.43214152 0.1879262  0.1685513  0.86354442
  0.90106222 0.12495473 0.84478761 0.37434414 0.80557454 0.58683475
  0.5619356  0.92027078 0.77079573 0.09208879 0.74899769 0.26987587
  0.5340589  0.36836255 0.3805782  0.28823068 0.50237955 0.76859644
  0.31432891 0.31571898 0.48555238 0.86227317 0.00497683 0.1786809
  0.08135387 0.73771213 0.35645213 0.040478   0.90167316 0.97229126
  0.76927559 0.63268066 0.97657959 0.07714042]
 [0.97777293 0.59275634 0.68549796 0.31885991 0.55554466 0.13193252
  0.8331448  0.42421244 0.09062871 0.68147527 0.04154347 0.60915665
  0.88615677 0.88448256 0.43445509 0.92869567 0.3681505  0.96834433
  0.16672366 0.36822984 0.6504041

In [112]:
print("Step 9: We assume we trained the 8 heads of the attention sub-layer")
z0h1 = np.random.random((3, 64))
z1h2 = np.random.random((3, 64))
z2h3 = np.random.random((3, 64))
z3h4 = np.random.random((3, 64))
z4h5 = np.random.random((3, 64))
z5h6 = np.random.random((3, 64))
z6h7 = np.random.random((3, 64))
z7h8 = np.random.random((3, 64))
print("shape of one head", z0h1.shape, "dimension of 8 heads", 64 * 8)

Step 9: We assume we trained the 8 heads of the attention sub-layer
shape of one head (3, 64) dimension of 8 heads 512


In [114]:
print(    "Step 10: Concatenate heads 1 to 8 to get the original 8x64=512 output dim")
output_attention = np.hstack(
    (z0h1, z1h2, z2h3, z3h4, z4h5, z5h6, z6h7, z7h8)
)
print(output_attention.shape)
print(output_attention)

Step 10: Concatenate heads 1 to 8 to get the original 8x64=512 output dim
(3, 512)
[[0.25916597 0.37829346 0.51030729 ... 0.43068194 0.42608497 0.53899957]
 [0.34347114 0.74826045 0.19870525 ... 0.07026327 0.97041318 0.67009303]
 [0.59149935 0.13513659 0.22579544 ... 0.41695596 0.87668181 0.09466595]]
